# SkillCraft Technology — Task 2: Data Cleaning and Preparation

This notebook demonstrates data cleaning with Python and Pandas. Upload your raw CSV when prompted. The included practice file contains intentional issues and is synthetic, not official SkillCraft data.

## Cleaning choices
- Trim whitespace and standardize Region and Category labels.
- Convert mixed date strings to dates; unparseable dates become missing.
- Convert Sales, Profit, and Quantity to numeric values, including currency strings.
- Remove exact duplicate rows.
- Drop rows missing critical fields (Order ID, Order Date, Category, Sales).
- Replace missing Region/Product labels with `Unknown`.
- Drop rows with missing/invalid Profit or Quantity rather than inventing financial values.
- Export the cleaned result to a new CSV.


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# Upload your raw CSV file when prompted
uploaded = files.upload()
input_file = next(iter(uploaded.keys()))
df = pd.read_csv(input_file)

print('Original shape:', df.shape)
display(df.head())
print('\nMissing values by column:\n', df.isna().sum())
print('\nExact duplicate rows:', df.duplicated().sum())

In [ ]:
# 1) Standardize text fields
for col in ['Order ID', 'Region', 'Category', 'Product']:
    df[col] = df[col].astype('string').str.strip()
df['Region'] = df['Region'].str.title()
df['Category'] = df['Category'].str.title()

# 2) Convert mixed date formats to datetime
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce', dayfirst=True)

# 3) Convert money/quantity fields to numeric; remove currency symbols and commas first
for col in ['Sales', 'Profit', 'Quantity']:
    cleaned = (df[col].astype('string').str.strip()
               .str.replace(r'[$₹,]', '', regex=True)
               .replace({'': pd.NA, 'N/A': pd.NA, 'NA': pd.NA, 'three': pd.NA}))
    df[col] = pd.to_numeric(cleaned, errors='coerce')

# 4) Remove exact duplicate rows
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates()

# 5) Drop rows missing essential fields
rows_before_required = len(df)
df = df.dropna(subset=['Order ID', 'Order Date', 'Category', 'Sales'])
critical_rows_removed = rows_before_required - len(df)

# 6) Handle missing labels without inventing business values
df['Region'] = df['Region'].fillna('Unknown').replace('', 'Unknown')
df['Product'] = df['Product'].fillna('Unknown').replace('', 'Unknown')

# 7) Drop rows with missing Profit/Quantity because these are required numeric measures
rows_before_financial = len(df)
df = df.dropna(subset=['Profit', 'Quantity'])
financial_rows_removed = rows_before_financial - len(df)
df['Quantity'] = df['Quantity'].astype(int)
df['Sales'] = df['Sales'].round(2)
df['Profit'] = df['Profit'].round(2)
df = df.sort_values('Order Date').reset_index(drop=True)

print('Duplicates removed:', duplicates_removed)
print('Rows removed for missing critical fields:', critical_rows_removed)
print('Rows removed for missing Profit/Quantity:', financial_rows_removed)
print('Cleaned shape:', df.shape)
print('\nMissing values after cleaning:\n', df.isna().sum())
display(df.head())

In [ ]:
# 8) Export cleaned data as a new CSV and download it
output_file = 'SkillCraft_Task2_Cleaned_Sales_Data.csv'
df.to_csv(output_file, index=False, date_format='%Y-%m-%d')
files.download(output_file)
print('Exported:', output_file)